# 확률 분포와 손실 함수

손실 함수는 모델 출력이 관측값을 얼마나 그럴듯하게 설명하는지를 음의 로그우도로 표현한 것이다. 회귀의 MSE는 정규분포 잡음을 가정한 최대우도추정과 연결되고, 분류의 Cross-Entropy는 베르누이 또는 카테고리 분포의 최대우도추정과 연결된다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def normal_pdf(x, mean, std):
    return (1 / (std * np.sqrt(2 * np.pi))) * np.exp(-((x - mean) ** 2) / (2 * std ** 2))

x = np.linspace(-4, 5, 400)
plt.figure(figsize=(7, 4))
plt.plot(x, normal_pdf(x, 0, 1), label='N(0, 1)')
plt.plot(x, normal_pdf(x, 2, 0.5), label='N(2, 0.5)')
plt.title('Normal PDFs')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
def bernoulli_pmf(p):
    return np.array([1 - p, p])

x_pos = np.array([0, 1])
width = 0.34
plt.figure(figsize=(6, 4))
plt.bar(x_pos - width / 2, bernoulli_pmf(0.3), width=width, label='B(0.3)')
plt.bar(x_pos + width / 2, bernoulli_pmf(0.7), width=width, label='B(0.7)')
plt.xticks(x_pos)
plt.ylim(0, 1)
plt.title('Bernoulli PMFs')
plt.legend()
plt.grid(axis='y', alpha=0.25)
plt.show()

In [ ]:
def softmax(logits):
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)

prob = softmax(np.array([1.0, 2.0, 3.0]))
print(prob)
print('sum gap:', abs(np.sum(prob) - 1.0))

## MSE 최소화와 정규분포 MLE

회귀 모델을 `y_i = f(x_i; theta) + epsilon_i`로 두고, 오차가 `epsilon_i ~ N(0, sigma^2)`를 따른다고 가정한다.

따라서 조건부 확률은 다음과 같다.

`p(y_i | x_i, theta) = 1 / sqrt(2 pi sigma^2) * exp(-(y_i - f(x_i; theta))^2 / (2 sigma^2))`

독립 표본의 로그우도는

`log L(theta) = sum_i log p(y_i | x_i, theta)`

`= sum_i [-log sqrt(2 pi sigma^2) - (y_i - f(x_i; theta))^2 / (2 sigma^2)]`

이다. `sigma`가 고정되어 있으면 첫 항과 `1 / (2 sigma^2)`는 최적화 관점에서 상수이므로

`argmax_theta log L(theta) = argmin_theta sum_i (y_i - f(x_i; theta))^2`

가 된다. 표본 수 `n`으로 나누면 MSE이므로, MSE 최소화는 정규분포 잡음 가정 아래의 MLE와 같은 해를 갖는다.

## Cross-Entropy 최소화와 베르누이/카테고리 MLE

이진 분류에서 모델이 `p_i = P(y_i=1 | x_i; theta)`를 출력한다고 두면 베르누이 우도는

`p(y_i | x_i, theta) = p_i^y_i * (1 - p_i)^(1 - y_i)`

이다. 음의 로그우도는

`-log L(theta) = -sum_i [y_i log p_i + (1-y_i) log(1-p_i)]`

이고, 이것이 이진 Cross-Entropy의 합이다. 따라서 이진 Cross-Entropy를 최소화하는 것은 베르누이 분포 가정 아래 MLE를 수행하는 것과 같다.

다중 분류에서 one-hot 정답을 `y_ik`, 소프트맥스 출력을 `p_ik`라고 하면 카테고리 분포의 우도는

`p(y_i | x_i, theta) = product_k p_ik^y_ik`

이고 음의 로그우도는

`-log L(theta) = -sum_i sum_k y_ik log p_ik`

이다. 이것이 다중 클래스 Cross-Entropy이므로, Cross-Entropy 최소화는 카테고리 분포 가정 아래의 MLE와 같은 목적 함수를 최적화한다.

## 정보 이론 보너스

엔트로피 `H(P) = -sum p log p`는 분포 자체의 불확실성을 측정한다. KL-Divergence `KL(P || Q) = sum p log(p/q)`는 참 분포 `P`를 예측 분포 `Q`로 대체할 때 생기는 추가 비용이다. Cross-Entropy `H(P, Q) = -sum p log q`는 `H(P) + KL(P || Q)`로 분해되므로, 참 분포가 고정되어 있을 때 Cross-Entropy 최소화는 KL-Divergence 최소화와 같다.